# Veritas-R1 — Stage 1 evaluation + calibration

Reads the per-seed logits saved by `veritas-stage1-train.ipynb`, computes:
- Per-seed: accuracy, NLL, macro-F1, **Brier**, **ECE-15**
- Per-seed reliability diagram
- **Temperature scaling** on each seed using its val logits
- **3-seed deep ensemble** (probability-space average)
- Ensemble + temperature-scaled vs ensemble alone vs each member alone
- Selective-prediction curve (accuracy vs coverage)

**Direct comparison target:** the original Stage-1 24-row run had confidence increasing while accuracy held flat. We want this run to show:
- ECE going **down** (less miscalibration)
- Accuracy going **up** vs base model
- Selective accuracy curve **monotone** (well-ordered uncertainty)

In [ ]:
%pip install --quiet --upgrade "matplotlib>=3.8" "scipy>=1.13"

In [ ]:
import json, math
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from scipy.special import softmax as np_softmax
import matplotlib.pyplot as plt

ADAPTER_ROOT = Path("/kaggle/working/adapters")
OUT_ROOT = Path("/kaggle/working")
RESULTS_DIR = OUT_ROOT / "eval_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ("SUPPORTS", "REFUTES", "NOT ENOUGH INFO")
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

# Discover available seeds.
seed_dirs = sorted(d for d in ADAPTER_ROOT.glob("seed_*") if (d / "logits.npz").is_file())
print(f"found {len(seed_dirs)} seeds")
for d in seed_dirs: print("  ", d.name)

## 1. Calibration metrics

In [ ]:
def accuracy(probs, labels):
    return float((probs.argmax(axis=1) == labels).mean())

def macro_f1(probs, labels):
    preds = probs.argmax(axis=1); n_c = probs.shape[1]; f1s = []
    for c in range(n_c):
        tp = int(((preds == c) & (labels == c)).sum())
        fp = int(((preds == c) & (labels != c)).sum())
        fn = int(((preds != c) & (labels == c)).sum())
        if tp + fp == 0 or tp + fn == 0: f1s.append(0.0); continue
        p = tp / (tp + fp); r = tp / (tp + fn)
        f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    return float(np.mean(f1s))

def brier_score(probs, labels):
    one_hot = np.zeros_like(probs); one_hot[np.arange(len(labels)), labels] = 1
    return float(((probs - one_hot) ** 2).sum(axis=1).mean())

def expected_calibration_error(probs, labels, n_bins=15):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1); correct = (pred == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0; n = len(labels)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == n_bins - 1 else (conf >= lo) & (conf < hi)
        if not mask.any(): continue
        ece += mask.sum() / n * abs(conf[mask].mean() - correct[mask].mean())
    return float(ece)

def reliability_bins(probs, labels, n_bins=15):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1); correct = (pred == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    mc = np.full(n_bins, np.nan); ma = np.full(n_bins, np.nan); cnt = np.zeros(n_bins, dtype=int)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == n_bins - 1 else (conf >= lo) & (conf < hi)
        cnt[i] = int(mask.sum())
        if cnt[i] > 0: mc[i] = float(conf[mask].mean()); ma[i] = float(correct[mask].mean())
    return edges, mc, ma, cnt

def selective_accuracy_curve(probs, labels, steps=21):
    conf = probs.max(axis=1); pred = probs.argmax(axis=1); correct = (pred == labels).astype(float)
    order = np.argsort(-conf); cs = correct[order]; n = len(labels)
    cov = np.linspace(0, 1, steps); acc = np.zeros(steps)
    for i, c in enumerate(cov):
        k = max(1, int(round(c * n))); acc[i] = float(cs[:k].mean())
    return cov, acc

def fit_temperature(logits, labels, max_iter=200):
    L = torch.as_tensor(logits, dtype=torch.float32); Y = torch.as_tensor(labels, dtype=torch.long)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.01, max_iter=max_iter, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(L / torch.exp(log_T), Y); loss.backward(); return loss
    opt.step(closure)
    return float(np.clip(float(torch.exp(log_T).item()), 0.1, 100.0))

def report_metrics(probs, labels, name):
    return {
        "name": name,
        "accuracy": round(accuracy(probs, labels), 4),
        "macro_f1": round(macro_f1(probs, labels), 4),
        "brier": round(brier_score(probs, labels), 4),
        "ece": round(expected_calibration_error(probs, labels), 4),
        "avg_confidence": round(float(probs.max(axis=1).mean()), 4),
    }

## 2. Per-seed metrics — uncalibrated, then temperature-scaled

In [ ]:
per_seed_test_probs = []
per_seed_test_probs_scaled = []
test_labels = None
scoreboard = []

for d in seed_dirs:
    data = np.load(d / "logits.npz")
    val_logits, val_labels = data["val_logits"], data["val_labels"]
    test_logits, t_labels = data["test_logits"], data["test_labels"]
    if test_labels is None: test_labels = t_labels
    else: assert np.array_equal(test_labels, t_labels), "all seeds must share the test set"
    
    raw_probs = np_softmax(test_logits, axis=-1)
    raw_metrics = report_metrics(raw_probs, test_labels, f"{d.name} raw")
    
    T = fit_temperature(val_logits, val_labels)
    scaled_probs = np_softmax(test_logits / T, axis=-1)
    scaled_metrics = report_metrics(scaled_probs, test_labels, f"{d.name} T={T:.3f}")
    
    per_seed_test_probs.append(raw_probs)
    per_seed_test_probs_scaled.append(scaled_probs)
    scoreboard.append(raw_metrics)
    scoreboard.append(scaled_metrics)
    print(f"{d.name} | T={T:.3f} | raw acc={raw_metrics['accuracy']:.4f} ece={raw_metrics['ece']:.4f} -> scaled ece={scaled_metrics['ece']:.4f}")

## 3. Deep-ensemble (probability-space average)

In [ ]:
ens_probs_raw = np.mean(np.stack(per_seed_test_probs, axis=0), axis=0)
ens_probs_scaled = np.mean(np.stack(per_seed_test_probs_scaled, axis=0), axis=0)
scoreboard.append(report_metrics(ens_probs_raw, test_labels, "ensemble (raw avg)"))
scoreboard.append(report_metrics(ens_probs_scaled, test_labels, "ensemble (T-scaled then avg)"))

print("\n=== scoreboard ===")
for s in scoreboard:
    print(f"  {s['name']:<35} acc={s['accuracy']:.4f} f1={s['macro_f1']:.4f} ece={s['ece']:.4f} brier={s['brier']:.4f} conf={s['avg_confidence']:.3f}")
(RESULTS_DIR / "scoreboard.json").write_text(json.dumps(scoreboard, indent=2))

## 4. Reliability diagrams

In [ ]:
def plot_reliability(probs, labels, title, ax):
    edges, mc, ma, cnt = reliability_bins(probs, labels)
    centers = 0.5 * (edges[1:] + edges[:-1])
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label="perfect")
    valid = ~np.isnan(mc)
    ax.bar(centers[valid], ma[valid], width=1.0/len(centers), edgecolor="black", alpha=0.7, label="observed")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("confidence"); ax.set_ylabel("empirical accuracy")
    ax.set_title(title); ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
plot_reliability(per_seed_test_probs[0], test_labels, "seed 17 (raw)", axes[0, 0])
plot_reliability(per_seed_test_probs_scaled[0], test_labels, "seed 17 (T-scaled)", axes[0, 1])
plot_reliability(ens_probs_raw, test_labels, "ensemble (raw avg)", axes[1, 0])
plot_reliability(ens_probs_scaled, test_labels, "ensemble (T-scaled then avg)", axes[1, 1])
plt.tight_layout(); plt.savefig(RESULTS_DIR / "reliability.png", dpi=140); plt.show()

## 5. Selective-prediction curve

A well-calibrated classifier shows a **monotone** curve — accuracy increases as we abstain on the low-confidence tail.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for probs, name, style in [
    (per_seed_test_probs[0], "seed 17 raw", "--"),
    (per_seed_test_probs_scaled[0], "seed 17 T-scaled", "-"),
    (ens_probs_scaled, "ensemble (T-scaled then avg)", "-"),
]:
    cov, acc = selective_accuracy_curve(probs, test_labels)
    ax.plot(cov, acc, style, label=name, linewidth=2)
ax.set_xlim(0, 1); ax.set_ylim(0.5, 1.01); ax.invert_xaxis()
ax.set_xlabel("coverage (top-c% most-confident kept)"); ax.set_ylabel("accuracy")
ax.set_title("Selective-prediction curve"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "selective.png", dpi=140); plt.show()

## 6. Confusion matrix — best stack (ensemble, T-scaled then averaged)

In [ ]:
preds = ens_probs_scaled.argmax(axis=1)
cm = np.zeros((len(LABELS), len(LABELS)), dtype=int)
for t, p in zip(test_labels, preds): cm[t, p] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black" if cm[i, j] < cm.max()/2 else "white")
ax.set_xticks(range(len(LABELS))); ax.set_yticks(range(len(LABELS)))
ax.set_xticklabels(LABELS); ax.set_yticklabels(LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("confusion matrix — ensemble (T-scaled, prob-avg)")
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=140); plt.show()
np.save(RESULTS_DIR / "confusion_matrix.npy", cm)

## 7. Final summary written to disk

In [ ]:
summary = {
    "scoreboard": scoreboard,
    "best_stack": "ensemble (T-scaled then avg)",
    "best_metrics": scoreboard[-1],
    "reliability_png": str(RESULTS_DIR / "reliability.png"),
    "selective_png": str(RESULTS_DIR / "selective.png"),
    "confusion_png": str(RESULTS_DIR / "confusion_matrix.png"),
}
(OUT_ROOT / "stage1_eval_summary.json").write_text(json.dumps(summary, indent=2))
print("saved summary to", OUT_ROOT / "stage1_eval_summary.json")